# AfriVoices — GRILLE BETA ÉTENDUE (segmentation) sur v8-cible

**Hypothèse** : le WER vient surtout de la SEGMENTATION de mots (CER=0.094 mais WER=0.342).
Le `beta` de pyctcdecode contrôle l'insertion de mots/frontières. On teste beta large
{0.5 → 5.0} à alpha=0.7, sur le dev officiel, pour voir si un beta plus élevé (plus de
frontières) aligne mieux la sortie sur la référence.

**Rapide** : on met les logits du dev en cache (une passe GPU), puis la grille ne fait
que redécoder. À exécuter dans une session avec le dev déjà reconstruit (eval_items),
OU ce notebook le reconstruit lui-même (§2).

Décision : si un beta baisse le WER dev de v8 sous 0.3178 → resoumettre avec ce beta.

## 1. Dépendances (redémarrer si fraîchement installé)

In [ ]:
import importlib, subprocess
need=[m for m in ["kenlm","pyctcdecode","jiwer"] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run(["pip","install","-q","pyctcdecode","jiwer"], check=False)
    subprocess.run(["pip","install","-q","https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("⚠️ REDÉMARRE le runtime puis reprends en §2")
else:
    print("✓ prêt")

## 2. Modèle + dev + logits en cache (une seule passe GPU)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, glob, re, io, base64, numpy as np, soundfile as sf, librosa, torch
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor

BASE="/content/drive/MyDrive/afrivoices"
MODEL=f"{BASE}/baobab-asr-v8-cible"
LM=f"{BASE}/lm_v2"
dev="cuda" if torch.cuda.is_available() else "cpu"

model=Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(dev).eval()
processor=Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)
tokenizer=processor.tokenizer

def clean_text(t):
    t=(t or "").lower().strip()
    t=re.sub(r"[^\w\s\u0129\u0169\u00e1\u00e0\u00e2\u00e4\u00e9\u00e8\u00ea\u00eb\u00ed\u00ec\u00ee\u00ef\u00f3\u00f2\u00f4\u00f6\u00fa\u00f9\u00fb\u00fc\']","",t)
    return re.sub(r"\s+"," ",t)

def duree_bytes(b):
    try: i=sf.info(io.BytesIO(b)); return i.frames/i.samplerate
    except Exception:
        try: i=sf.info(io.BytesIO(base64.b64decode(b))); return i.frames/i.samplerate
        except Exception: return 999.0

def decode_bytes(b):
    try: arr,sr=sf.read(io.BytesIO(b),dtype="float32")
    except Exception: arr,sr=sf.read(io.BytesIO(base64.b64decode(b)),dtype="float32")
    if arr.ndim>1: arr=arr.mean(axis=1)
    if sr!=16000: arr=librosa.resample(arr,orig_sr=sr,target_sr=16000)
    return arr.astype(np.float32)

ANVKE=f"{BASE}/anvke"
ALIAS={"kik":["kik","kikuyu","gikuyu"],"luo":["luo","dholuo"],"kln":["kln","kalenjin"],
       "mas":["mas","maasai"],"som":["som","somali"]}
lang_dirs={}
for d in os.listdir(ANVKE):
    dl=d.lower()
    for lang,als in ALIAS.items():
        if any(a in dl for a in als) and lang not in lang_dirs: lang_dirs[lang]=os.path.join(ANVKE,d)

from datasets import load_dataset, Audio
logits_cache={}   # lang -> [(logits, ref), ...]
with torch.no_grad():
    for lang,root in lang_dirs.items():
        allp=sorted(glob.glob(f"{root}/**/*.parquet", recursive=True))
        dv=[p for p in allp if "/dev/" in p or os.path.basename(p).startswith("dev")]
        picks=[p for p in dv if "unscripted" not in os.path.basename(p)][:1] + \
              [p for p in dv if "unscripted" in os.path.basename(p)][:1]
        if not picks: picks=dv[:2] if dv else allp[:1]
        d=load_dataset("parquet", data_files=picks, split="train").cast_column("audio", Audio(decode=False))
        rows=[]
        for ex in d.shuffle(seed=42):
            if len(rows)>=60: break
            t=(ex.get("transcription") or "").strip()
            b=ex["audio"].get("bytes") if isinstance(ex["audio"],dict) else None
            if not t or not b or duree_bytes(b)>20: continue
            arr=decode_bytes(b)
            fe=processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt", padding=True)
            lg=model(**{k:v.to(dev) for k,v in fe.items()}).logits[0].cpu().numpy()
            rows.append((lg, clean_text(t)))
        logits_cache[lang]=rows
        print(f"{lang}: {len(rows)} logits en cache")

## 3. Grille beta étendue (alpha fixé à 0.7)

In [ ]:
from pyctcdecode import build_ctcdecoder
from collections import Counter
import jiwer

raw=[t for t,_ in sorted(tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank=tokenizer.pad_token
labels=[]; n=0
for t in raw:
    if t==blank: labels.append("")
    elif t in ("|"," "): labels.append(" ")
    elif t in {"[UNK]","<s>","</s>"}: n+=1; labels.append("\u2047"*n)
    else: labels.append(t)

def unigrams(lang, top=50000):
    c=Counter()
    for line in open(f"{LM}/{lang}.txt", encoding="utf-8"): c.update(line.split())
    return [w for w,_ in c.most_common(top)]
UNI={l: unigrams(l) for l in logits_cache}

ALPHA=0.7
BETAS=[0.5, 1.0, 2.0, 3.0, 4.0, 5.0]

print(f"{'beta':>5} | " + " ".join(f"{l:>6}" for l in logits_cache) + " |  macro")
best=(None, 9.9)
results={}
for b in BETAS:
    per={}
    for lang, rows in logits_cache.items():
        dec=build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin",
                             unigrams=UNI[lang], alpha=ALPHA, beta=b)
        preds=[dec.decode(lg) for lg,_ in rows]
        refs=[r for _,r in rows]
        per[lang]=jiwer.wer(refs,preds)
    macro=sum(per.values())/len(per)
    results[b]=(per,macro)
    if macro<best[1]: best=(b,macro)
    print(f"{b:5.1f} | " + " ".join(f"{per[l]:6.3f}" for l in logits_cache) + f" | {macro:.4f}")

print(f"\nMEILLEUR beta={best[0]} -> macro {best[1]:.4f}  (référence v8 beta=0.5 : 0.3178)")
if best[1] < 0.3178:
    print(f"✅ GAIN de {0.3178-best[1]:.4f} sur le dev -> resoumettre avec beta={best[0]}")
    print(f"   estimation leaderboard: ~{best[1]+0.10:.3f} (vs 0.417 actuel)")
else:
    print("❌ aucun beta ne bat 0.3178 -> la segmentation n'est pas récupérable par beta.")
    print("   0.417 est proche du plafond de cette architecture. Cap sur le dépôt conforme.")

## 4. Si un beta gagne : affiner autour (optionnel)

Si par ex. beta=3.0 est le meilleur, teste {2.5, 3.0, 3.5} en changeant BETAS ci-dessus
et relance §3 (les logits sont en cache, c'est instantané). Puis resoumets avec le
générateur de soumission simple : MODEL_NAME=baobab-asr-v8-cible, ALPHA=0.7, BETA=<gagnant>.

Rappelle-toi : l'optimum dev est bruité (±0.01 sur ~300 ex). Un gain net (>0.01) vaut la
soumission ; un micro-gain (<0.005) est probablement du bruit.